# Graph features for the tree model

The GNN ablation showed that the true flow graph carries a small real signal (about +0.015 macro F1 over a shuffled graph), but the GNN's neural classifier is weaker than LightGBM. Here the graph signal is handed to the strong classifier instead: for every flow we add **graph features** (how busy its two endpoints are, and the average of the *other* flows at each endpoint) to the 45 flow features and train the same two-stage LightGBM.

| Condition | Features |
|---|---|
| **flow features only** | the 45 preprocessed features (control) |
| **+ graph features** | 45 + endpoint degrees + neighbour feature means, from the real graph |
| **+ graph features, shuffled edges** | the same extra columns computed on a graph whose edges were shuffled (flows get another flow's endpoints) |

The last row is the important control: if shuffled-edge features help as much as real ones, the gain would come from having extra columns, not from the graph.

**No leakage and matching density.** Graph features use flow *features* only, never labels. The train graph is split into random chunks of the size of the test set, and features are computed inside each chunk, so train and test flows see graphs of the same density (a graph of all train flows has about 4x more neighbours per endpoint than the test graph, which would make the features shift between train and test). The test graph is built from test flows only. SMOTE rows have no endpoints, so every condition trains on the original `train.parquet` (class-weighted instead of resampled; resampling did not change results earlier).

Every condition is repeated with 3 seeds (validation split and model seed), the same seeds for every condition.

In [ ]:
import glob

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

DEBUG = False   # True = quick smoke test on a small random sample
SEEDS = [42] if DEBUG else [42, 43, 44]
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"

## 1. Load data

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=0), test_df.sample(30_000, random_state=0)
train_graph = pd.read_csv(f"{DATASET_PATH}/train_graph.csv", index_col="row_id").loc[train_df.index]
test_graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
normal_label = list(class_names).index("Normal")
attack_ids = [i for i in range(n_classes) if i != normal_label]
remap = {orig: new for new, orig in enumerate(attack_ids)}
flow_cols = [c for c in train_df.columns if c != "label"]
print("train flows", len(train_df), "| test flows", len(test_df), "| flow features", len(flow_cols))

## 2. Graph features

For one graph (nodes = (IP, port) endpoints, edges = flows), every flow gets:
- `deg_src`, `deg_dst`: how many flows touch its source / destination endpoint;
- `src_mean_<f>`, `dst_mean_<f>`: the mean of feature `f` over the **other** flows at that endpoint (missing when the endpoint has no other flow; LightGBM handles missing values).

Sums per endpoint come from a sparse incidence matrix, so this is a handful of matrix products.

In [ ]:
def graph_features(X, graph_df):
    n = len(X)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, node_keys = pd.factorize(ip_code.astype("int64") * 70_000 + ports)
    src, dst = node_id[:n], node_id[n:]
    edges = np.arange(n)
    ones = np.ones(n, dtype="float32")
    incidence = (csr_matrix((ones, (src, edges)), shape=(len(node_keys), n))
                 + csr_matrix((ones, (dst, edges)), shape=(len(node_keys), n)))
    degree = np.asarray(incidence.sum(1)).ravel()
    node_sum = incidence @ X                       # (nodes, features): sum over all flows at each endpoint

    out = {"deg_src": degree[src], "deg_dst": degree[dst]}
    for side, node in [("src", src), ("dst", dst)]:
        others = degree[node] - 1
        mean = (node_sum[node] - X) / np.where(others > 0, others, np.nan)[:, None]
        out.update({f"{side}_mean_{f}": mean[:, j] for j, f in enumerate(flow_cols)})
    return pd.DataFrame(out, index=graph_df.index).astype("float32")


def shuffle_endpoints(graph_df, seed):
    """Give every flow another flow's endpoints (the four columns move together)."""
    perm = np.random.default_rng(seed).permutation(len(graph_df))
    out = graph_df.copy()
    for col in ["srcip", "dstip", "sport", "dsport"]:
        out[col] = graph_df[col].to_numpy()[perm]
    return out


def features_for(graph_df, feat_df, shuffle, seed, n_chunks):
    """Graph features for all flows. Train: computed inside random chunks the size of the test set."""
    X = feat_df[flow_cols].to_numpy("float32")
    if shuffle:
        graph_df = shuffle_endpoints(graph_df, seed)
    parts = []
    chunk_of = np.random.default_rng(seed).integers(0, n_chunks, len(graph_df))
    for k in range(n_chunks):
        rows = np.where(chunk_of == k)[0]
        parts.append(graph_features(X[rows], graph_df.iloc[rows]))
    return pd.concat(parts).loc[feat_df.index]


n_chunks = max(1, round(len(train_df) / len(test_df)))
print("train chunks:", n_chunks)
demo = features_for(test_graph, test_df, False, 0, 1)
print(demo.shape, "| test flows with no other flow at the source endpoint: %.1f%%" % (100 * demo["src_mean_" + flow_cols[0]].isna().mean()))
demo[["deg_src", "deg_dst"]].describe().round(1)

## 3. Two-stage LightGBM

The same two-stage model as the baseline: Stage 1 separates attack from Normal, Stage 2 (attack rows only) picks the attack type; same hyper-parameters. Early stopping uses a validation slice of the training data; the test set is only used at the end.

In [ ]:
def run_two_stage(X_train, y_train, X_test, seed):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=seed)
    params = dict(class_weight="balanced", learning_rate=0.08, num_leaves=63,
                  n_estimators=300, random_state=seed, verbose=-1)
    stop = [lgb.early_stopping(20), lgb.log_evaluation(0)]

    stage1 = lgb.LGBMClassifier(objective="binary", **params)
    stage1.fit(X_tr, (y_tr != normal_label).astype(int),
               eval_set=[(X_val, (y_val != normal_label).astype(int))], callbacks=stop)

    a_tr, a_val = (y_tr != normal_label).to_numpy(), (y_val != normal_label).to_numpy()
    stage2 = lgb.LGBMClassifier(objective="multiclass", num_class=len(attack_ids), **params)
    stage2.fit(X_tr[a_tr], y_tr[a_tr].map(remap),
               eval_set=[(X_val[a_val], y_val[a_val].map(remap))], callbacks=stop)

    pred = np.full(len(X_test), normal_label)
    flagged = np.where(stage1.predict(X_test) == 1)[0]
    pred[flagged] = np.array(attack_ids)[stage2.predict(X_test.iloc[flagged])]
    return pred


CONDITIONS = {
    "flow features only": None,
    "+ graph features": False,
    "+ graph features, shuffled edges": True,
}
y_train, y_test = train_df["label"], test_df["label"].to_numpy()

## 4. Run all conditions

In [ ]:
import time

rows = []
for seed in SEEDS:
    for name, shuffle in CONDITIONS.items():
        t0 = time.time()
        if shuffle is None:
            X_train, X_test = train_df[flow_cols], test_df[flow_cols]
        else:
            X_train = pd.concat([train_df[flow_cols], features_for(train_graph, train_df, shuffle, seed, n_chunks)], axis=1)
            X_test = pd.concat([test_df[flow_cols], features_for(test_graph, test_df, shuffle, seed + 1000, 1)], axis=1)
        pred = run_two_stage(X_train, y_train, X_test, seed)
        per_class = f1_score(y_test, pred, labels=np.arange(n_classes), average=None, zero_division=0)
        rows.append({"condition": name, "seed": seed, "test macro F1": per_class.mean(),
                     **{f"F1 {c}": f for c, f in zip(class_names, per_class)}})
        print(f"{name:<34} seed {seed} | features {X_train.shape[1]:>3} | test macro F1 {per_class.mean():.4f} | {time.time() - t0:.0f}s", flush=True)
results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/graph_feature_runs.csv", index=False)

## 5. Results

Bars = mean over seeds, dots = individual seeds, dashed line = the baseline two-stage model trained on the SMOTE data (reference only; the conditions here train on the original data).

In [ ]:
tab_file = glob.glob("/kaggle/input/**/two_stage_test_proba.parquet", recursive=True)
ref = None
if tab_file and not DEBUG:
    tab = pd.read_parquet(tab_file[0]).loc[test_df.index]
    ref = f1_score(y_test, tab.to_numpy().argmax(1), average="macro")

summary = results.groupby("condition", sort=False)["test macro F1"].agg(["mean", "std"])
print(summary.round(4).to_string())
if ref is not None:
    print("baseline two-stage (SMOTE training data): %.4f" % ref)

order = list(CONDITIONS)
plt.figure(figsize=(10, 3.8))
sns.barplot(data=results, y="condition", x="test macro F1", order=order, errorbar="sd", capsize=0.15, color="steelblue", alpha=0.8)
sns.stripplot(data=results, y="condition", x="test macro F1", order=order, color="black", size=5)
if ref is not None:
    plt.axvline(ref, color="darkorange", ls="--", label=f"baseline two-stage, SMOTE data ({ref:.3f})")
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.xlim(0.5, 0.7)
plt.title("Two-stage LightGBM with graph features (mean and std over seeds)")
plt.tight_layout()
plt.show()

p = results.pivot(index="seed", columns="condition", values="test macro F1")
for control in ["flow features only", "+ graph features, shuffled edges"]:
    d = p["+ graph features"] - p[control]
    print(f"real graph features minus '{control}': per seed {d.round(4).tolist()} | mean {d.mean():+.4f}")

## 6. Per class

Left: mean test F1 per class. Right: real graph features minus the control without graph features, and minus the shuffled-edge version (mean over seeds, error bars = std of the paired difference).

In [ ]:
class_cols = [f"F1 {c}" for c in class_names]
mean_pc = results.groupby("condition", sort=False)[class_cols].mean()
mean_pc.columns = class_names
print(mean_pc.loc[order].round(3).T.to_string())

real = results[results.condition == "+ graph features"].set_index("seed")[class_cols]
rel = {}
for control in ["flow features only", "+ graph features, shuffled edges"]:
    d = real - results[results.condition == control].set_index("seed")[class_cols]
    d.columns = class_names
    rel[control] = d
means = pd.DataFrame({"vs " + k: v.mean() for k, v in rel.items()})
stds = pd.DataFrame({"vs " + k: v.std() for k, v in rel.items()})
print(means.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [3, 2]})
sns.heatmap(mean_pc.loc[order].T, annot=True, fmt=".2f", cmap="viridis", cbar=False, ax=axes[0])
axes[0].set_title("Mean test F1 per class")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)
means.plot.barh(ax=axes[1], xerr=stds.to_numpy().T, capsize=2, width=0.75)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Gain from real graph features (F1 difference)")
plt.tight_layout()
plt.show()